# Demo: Blunder Analysis

This notebook demonstrates the `analyze_game_blunders` function from `src/lichess_analyser/engine/report.py`.
It creates a small synthetic game, provides a matching sequence of centipawn evaluations (one per ply), and shows how blunders and the game-losing blunder are detected and labelled with the game phase using `StockfishEngine.get_game_phase`.

In [13]:
# Setup: import the library code
import chess
import chess.pgn
from lichess_analyser.mistake_report import analyze_mistakes
from lichess_analyser.engine.engine import StockfishEngine

from lichess_analyser.game_loader import LichessGameLoader

# Lichess client

In [1]:
from src.lichess_analyser.lichess_client import LichessClient
from dotenv import load_dotenv
import os
import json
load_dotenv()
lichess_token = os.getenv("LICHESS_TOKEN")

In [2]:
client = LichessClient(token=lichess_token)

In [19]:
json_object = client.get_single_game("IaTSe0hv", params = {"pgnInJson": "true"})

In [20]:
json_object

'{"id":"IaTSe0hv","rated":true,"variant":"standard","speed":"blitz","perf":"blitz","createdAt":1760868539234,"lastMoveAt":1760868945241,"status":"resign","source":"pool","players":{"white":{"user":{"name":"Thijs1337","id":"thijs1337"},"rating":1637,"ratingDiff":-6},"black":{"user":{"name":"a1architects","id":"a1architects"},"rating":1624,"ratingDiff":6}},"fullId":"IaTSe0hvPg4L","winner":"black","opening":{"eco":"E61","name":"King\'s Indian Defense","ply":5},"moves":"d4 Nf6 c4 g6 Nc3 Bg7 Nf3 d6 Bf4 O-O e3 Bg4 Be2 c6 O-O a6 h3 Bd7 Rc1 Re8 a3 Nh5 Bh2 f5 b4 Be6 d5 Bf7 Na4 b5 Nc3 bxc4 dxc6 Nxc6 Nd4 Nxd4 exd4 d5 Na4 Nf6 Nc5 Ne4 Nb7 Qb6 Nc5 a5 Rb1 axb4 Nxe4 fxe4 Rxb4 Qxd4 Qb1 Rxa3 Rb8 Ra8 Rxe8+ Rxe8 Qb7 e3 Rb1 exf2+ Kf1 Qe3 Qd7 d4 Bg3 d3 Bxf2 Qxe2+ Kg1 c3 Re1 Qc2 Rxe7 Rxe7 Qxe7 Bf8 Qd8 d2 Bc5 d1=Q+ Qxd1 Qxd1+ Kh2","clocks":[18003,18003,18147,18123,18307,18179,18379,18331,18483,18355,18611,18379,18675,18387,18803,18435,18931,18443,18771,18547,18883,18243,18843,18299,18851,18331,18147,18387,158

In [21]:
game = json.loads(json_object)
game.keys()

dict_keys(['id', 'rated', 'variant', 'speed', 'perf', 'createdAt', 'lastMoveAt', 'status', 'source', 'players', 'fullId', 'winner', 'opening', 'moves', 'clocks', 'pgn', 'clock', 'division'])

In [22]:
game.get('pgn')

'[Event "rated blitz game"]\n[Site "https://lichess.org/IaTSe0hv"]\n[Date "2025.10.19"]\n[White "Thijs1337"]\n[Black "a1architects"]\n[Result "0-1"]\n[GameId "IaTSe0hv"]\n[UTCDate "2025.10.19"]\n[UTCTime "10:08:59"]\n[WhiteElo "1637"]\n[BlackElo "1624"]\n[WhiteRatingDiff "-6"]\n[BlackRatingDiff "+6"]\n[Variant "Standard"]\n[TimeControl "180+2"]\n[ECO "E61"]\n[Opening "King\'s Indian Defense"]\n[Termination "Normal"]\n\n1. d4 { [%clk 0:03:00] } 1... Nf6 { [%clk 0:03:00] } 2. c4 { [%clk 0:03:01] } 2... g6 { [%clk 0:03:01] } 3. Nc3 { [%clk 0:03:03] } 3... Bg7 { [%clk 0:03:02] } 4. Nf3 { [%clk 0:03:04] } 4... d6 { [%clk 0:03:03] } 5. Bf4 { [%clk 0:03:05] } 5... O-O { [%clk 0:03:04] } 6. e3 { [%clk 0:03:06] } 6... Bg4 { [%clk 0:03:04] } 7. Be2 { [%clk 0:03:07] } 7... c6 { [%clk 0:03:04] } 8. O-O { [%clk 0:03:08] } 8... a6 { [%clk 0:03:04] } 9. h3 { [%clk 0:03:09] } 9... Bd7 { [%clk 0:03:04] } 10. Rc1 { [%clk 0:03:08] } 10... Re8 { [%clk 0:03:05] } 11. a3 { [%clk 0:03:09] } 11... Nh5 { [%clk

In [26]:
import io
pgn_io = io.StringIO(game.get('pgn'))
pgn_game = chess.pgn.read_game(pgn_io)
pgn_game.headers

Headers(Event='rated blitz game', Site='https://lichess.org/IaTSe0hv', Date='2025.10.19', Round='?', White='Thijs1337', Black='a1architects', Result='0-1', GameId='IaTSe0hv', UTCDate='2025.10.19', UTCTime='10:08:59', WhiteElo='1637', BlackElo='1624', WhiteRatingDiff='-6', BlackRatingDiff='+6', Variant='Standard', TimeControl='180+2', ECO='E61', Opening="King's Indian Defense", Termination='Normal')

In [ ]:
# Evaluate the last game from the PGN using the StockfishEngine and analyze blunders
engine = StockfishEngine()
# Analyze the mainline positions to obtain per-ply centipawn evals.
# This will initialize and call the local Stockfish binary; it may take some time.
evals = engine.analyze_game_evals(pgn_game, depth=20, verbose= True)

In [ ]:

# Run the mistake analysis. Pass player_color or player_name; here we use player_name.
report = analyze_mistakes(pgn_game, evals, player_name='Thijs1337')

# Full list of player's moves with categories
report.get('all_moves')

# Moves where the opponent capitalized
report.get('capitalized')

# Losing move (if any)
report.get('losing_move')

In [ ]:
# Count the number of different categories in each 'all_moves' entry
from collections import Counter
category_counts = Counter()
for move_info in report.get('all_moves', []):
    category = move_info.get('category', 'unknown')
    category_counts[category] += 1
category_counts


In [ ]:
# Extract all the 'Mistake' moves
mistakes = [move for move in report.get('all_moves', []) if move['category'] in {
    'Mistake'
}]

In [ ]:
mistakes

In [ ]:
report.get('all_moves')